# MS-VAR: Markov-Switching Vector Autoregression

Neste notebook, estendemos a abordagem de regime-switching para sistemas **multivariados**.
O modelo **MS-VAR** (Krolzig, 1997) permite que os parametros de um VAR mudem
de acordo com um regime latente governado por uma cadeia de Markov.

**Motivacao**: Em macroeconomia, multiplas variaveis (PIB, inflacao, taxa de juros)
frequentemente mudam de comportamento simultaneamente. O MS-VAR captura essa
co-movimentacao entre regimes.

**Conteudo:**
1. Motivacao para MS-VAR
2. MS(2)-VAR(1): estimacao
3. Regimes e impulso-resposta (IRF por regime)
4. Comparacao com VAR linear
5. Previsao condicional ao regime

**Referencias:**
- Krolzig, H.-M. (1997). *Markov-Switching Vector Autoregressions*. Springer.
- Hamilton, J.D. (1989). *A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle*. Econometrica.
- Ehrmann, M., Ellison, M. & Valla, N. (2003). Regime-dependent impulse response functions in a Markov-switching VAR model. *Economics Letters*, 78(3), 295-299.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
from utils.data_generator import generate_ms_var
from utils.plot_helpers import (
    plot_regime_probabilities,
    plot_transition_matrix,
)

from archbox.regime import (
    MarkovSwitchingVAR,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

print('Bibliotecas carregadas com sucesso.')

## 1. Motivacao para MS-VAR

O modelo **VAR linear** assume que as relacoes entre variaveis sao **constantes** ao longo do tempo.
Porem, em muitas aplicacoes, essas relacoes mudam de acordo com o **estado da economia**:

- Em **expansao**: o multiplicador fiscal e menor, a curva de Phillips e mais plana
- Em **recessao**: choques se propagam de forma diferente, correlacoes mudam

O **MS-VAR** resolve isso permitindo que os parametros do VAR dependam de um regime $S_t$:

$$\mathbf{Y}_t = \boldsymbol{\mu}_{S_t} + \boldsymbol{\Phi}_{1,S_t} \mathbf{Y}_{t-1} + \cdots + \boldsymbol{\Phi}_{p,S_t} \mathbf{Y}_{t-p} + \boldsymbol{\epsilon}_t$$

onde $\boldsymbol{\epsilon}_t \sim N(\mathbf{0}, \boldsymbol{\Sigma}_{S_t})$ e $S_t$ segue uma cadeia de Markov.

### Taxonomia de Krolzig (1997)

| Sigla | O que muda entre regimes |
|-------|-------------------------|
| **MSI** | Intercepto $\boldsymbol{\mu}$ |
| **MSM** | Media $\boldsymbol{\mu}$ |
| **MSH** | Heteroscedasticidade $\boldsymbol{\Sigma}$ |
| **MSA** | Coeficientes autorregressivos $\boldsymbol{\Phi}$ |
| **MSIH** | Intercepto + Heteroscedasticidade |
| **MSMH** | Media + Heteroscedasticidade |

In [ ]:
# Gerar dados MS-VAR sinteticos
df = generate_ms_var(n=250, seed=57)
print(df.head(10))
print(f'\nDimensoes: {df.shape}')
print(f'Regimes: {df["true_regime"].value_counts().to_dict()}')

# Plotar as 2 series com sombreamento de regimes
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for i, var in enumerate(['y1', 'y2']):
    ax = axes[i]
    ax.plot(df['date'], df[var], color='black', lw=0.8)
    # Sombrear regime 2
    ax.fill_between(
        df['date'], df[var].min(), df[var].max(),
        where=df['true_regime'] == 2,
        alpha=0.15, color='red', label='Regime 2'
    )
    ax.fill_between(
        df['date'], df[var].min(), df[var].max(),
        where=df['true_regime'] == 1,
        alpha=0.10, color='green', label='Regime 1'
    )
    ax.set_title(f'Serie {var}', fontsize=12)
    ax.set_ylabel(var)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
plt.suptitle('Dados Bivariados MS(2)-VAR(1) Sinteticos', fontsize=14)
plt.tight_layout()
plt.show()

# Estatisticas descritivas por regime
for regime in [1, 2]:
    mask = df['true_regime'] == regime
    print(f"\nRegime {regime} (n={mask.sum()}):")
    print(f"  y1: media={df.loc[mask, 'y1'].mean():.3f}, std={df.loc[mask, 'y1'].std():.3f}")
    print(f"  y2: media={df.loc[mask, 'y2'].mean():.3f}, std={df.loc[mask, 'y2'].std():.3f}")

## 2. MS(2)-VAR(1)

Para um VAR bivariado com 2 regimes, o modelo e:

$$\begin{pmatrix} y_{1,t} \\ y_{2,t} \end{pmatrix} = \begin{pmatrix} \mu_{1,S_t} \\ \mu_{2,S_t} \end{pmatrix} + \begin{pmatrix} \phi_{11,S_t} & \phi_{12,S_t} \\ \phi_{21,S_t} & \phi_{22,S_t} \end{pmatrix} \begin{pmatrix} y_{1,t-1} \\ y_{2,t-1} \end{pmatrix} + \begin{pmatrix} \epsilon_{1,t} \\ \epsilon_{2,t} \end{pmatrix}$$

onde $\boldsymbol{\epsilon}_t \sim N(\mathbf{0}, \boldsymbol{\Sigma}_{S_t})$ e:

$$\boldsymbol{\Sigma}_{S_t} = \begin{pmatrix} \sigma^2_{11,S_t} & \sigma_{12,S_t} \\ \sigma_{12,S_t} & \sigma^2_{22,S_t} \end{pmatrix}$$

### Numero de parametros

Para MS(K)-VAR(p) com $n$ variaveis:
- Interceptos: $K \times n$
- Coeficientes AR: $K \times n^2 \times p$ (se switching) ou $n^2 \times p$ (se nao)
- Covariancias: $K \times \frac{n(n+1)}{2}$
- Transicao: $K(K-1)$

In [ ]:
# Preparar dados como array (T, n)
Y = df[['y1', 'y2']].values
print(f"Shape dos dados: {Y.shape}")

# Criar e ajustar MS(2)-VAR(1)
model = MarkovSwitchingVAR(
    endog=Y,
    k_regimes=2,
    order=1,
    switching_mean=True,
    switching_variance=True,
)
results = model.fit(method='em', maxiter=500, verbose=True)

print('\n' + results.summary())

# Examinar parametros por regime
print('\n' + '='*50)
print('Parametros por regime:')
print('='*50)
for regime_id, params in results.regime_params.items():
    print(f'\n--- Regime {regime_id} ---')
    for name, value in params.items():
        print(f'  {name}: {value:.4f}')

## 3. Regimes e impulso-resposta

Uma das principais vantagens do MS-VAR e poder calcular **funcoes impulso-resposta (IRF)**
especificas para cada regime. Isso revela como choques se propagam de forma diferente
dependendo do estado da economia.

Para cada regime $s$, a IRF e calculada usando a matriz de coeficientes $\boldsymbol{\Phi}_{1,s}$
e a decomposicao de Cholesky de $\boldsymbol{\Sigma}_s$:

$$\text{IRF}_s(h) = \boldsymbol{\Phi}_{1,s}^h \cdot \mathbf{P}_s$$

onde $\mathbf{P}_s$ e o fator de Cholesky tal que $\boldsymbol{\Sigma}_s = \mathbf{P}_s \mathbf{P}_s'$.

**Interpretacao**: um choque unitario na variavel $j$ tem efeito diferente sobre a variavel $i$
dependendo de qual regime esta ativo — refletindo a **assimetria** das respostas economicas.

In [ ]:
# Calcular e plotar IRF para cada regime
# Extrair matrizes Phi e Sigma por regime usando _unpack_var_params
n_vars = 2
horizonte = 12
var_names = ['y1', 'y2']
regime_colors = ['#2ecc71', '#e74c3c']
regime_labels_irf = ['Regime 0 (forte cross-effects)', 'Regime 1 (fraco cross-effects)']

# Calcular IRF para cada regime
irf_all = {}
for s in range(results.k_regimes):
    mu_s, phi_s, sigma_s = model._unpack_var_params(results.params, s)
    # Phi_1 e a matriz (n, n) de coeficientes VAR(1)
    Phi_1 = phi_s[:, :n_vars]  # primeira lag

    # Decomposicao de Cholesky de Sigma
    try:
        chol = np.linalg.cholesky(sigma_s)
    except np.linalg.LinAlgError:
        chol = np.eye(n_vars) * np.sqrt(np.diag(sigma_s))

    # IRF: impulso[h] = Phi_1^h @ chol
    irf = np.zeros((horizonte + 1, n_vars, n_vars))
    irf[0] = chol  # impacto imediato
    Phi_power = np.eye(n_vars)
    for h in range(1, horizonte + 1):
        Phi_power = Phi_power @ Phi_1
        irf[h] = Phi_power @ chol

    irf_all[s] = irf

# Plotar IRF: resposta de y_i a choque em y_j, para cada regime
fig, axes = plt.subplots(n_vars, n_vars, figsize=(14, 10))
fig.suptitle('Funcoes Impulso-Resposta por Regime', fontsize=14, y=1.02)

for i in range(n_vars):
    for j in range(n_vars):
        ax = axes[i, j]
        for s in range(results.k_regimes):
            ax.plot(range(horizonte + 1), irf_all[s][:, i, j],
                    color=regime_colors[s], lw=2, marker='o', markersize=3,
                    label=regime_labels_irf[s])
        ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
        ax.set_title(f'Resposta de {var_names[i]} a choque em {var_names[j]}', fontsize=11)
        ax.set_xlabel('Horizonte')
        ax.set_ylabel('Resposta')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("As IRFs mostram como os choques se propagam de forma diferente em cada regime.")
print("No regime com cross-effects fortes, choques em uma variavel afetam mais a outra.")

## 4. Comparacao com VAR linear

Para avaliar se a mudanca de regime e empiricamente relevante, comparamos o MS-VAR
com um **VAR linear simples** (sem regimes):

$$\mathbf{Y}_t = \boldsymbol{\mu} + \boldsymbol{\Phi}_1 \mathbf{Y}_{t-1} + \boldsymbol{\epsilon}_t$$

**Criterios de comparacao:**
- **AIC/BIC**: penalizam complexidade adicional do MS-VAR
- **Log-verossimilhanca**: MS-VAR sempre tera $\ell$ maior, mas nao necessariamente melhor fit ajustado
- **Qualidade da previsao**: comparar RMSE fora da amostra

O VAR linear e um caso especial do MS-VAR quando $K=1$ (um unico regime).

In [ ]:
# Comparacao com VAR linear simples (OLS direto)
# Estimacao via OLS para evitar k_regimes < 2
from numpy.linalg import lstsq

p_var = 1
Y_dep = Y[p_var:]  # (T-1, n)
Y_lag = Y[:-p_var]  # (T-1, n)
X_ols = np.column_stack([np.ones(len(Y_lag)), Y_lag])  # (T-1, 1+n)

# OLS: Y_dep = X_ols @ B + e
B_ols, residuals_ols, _, _ = lstsq(X_ols, Y_dep, rcond=None)
Y_hat = X_ols @ B_ols
resid_ols = Y_dep - Y_hat
Sigma_ols = (resid_ols.T @ resid_ols) / len(resid_ols)

# Log-likelihood do VAR linear
n_ols = len(Y_dep)
n_v = n_vars
log_det_sigma = np.log(np.linalg.det(Sigma_ols))
ll_ols = -0.5 * n_ols * (n_v * np.log(2 * np.pi) + log_det_sigma)
for t in range(n_ols):
    ll_ols -= 0.5 * resid_ols[t] @ np.linalg.inv(Sigma_ols) @ resid_ols[t]

k_params_ols = n_v * (1 + n_v * p_var) + n_v * (n_v + 1) // 2
aic_ols = -2 * ll_ols + 2 * k_params_ols
bic_ols = -2 * ll_ols + np.log(n_ols) * k_params_ols

# Comparacao
print('='*55)
print('Comparacao VAR linear vs MS(2)-VAR(1):')
print('='*55)
print(f'  VAR linear: AIC={aic_ols:.2f}, BIC={bic_ols:.2f}')
print(f'  MS(2)-VAR:  AIC={results.aic:.2f}, BIC={results.bic:.2f}')
print(f'  Log-lik VAR: {ll_ols:.2f}')
print(f'  Log-lik MS:  {results.loglike:.2f}')
print(f'  Params VAR: {k_params_ols}')
print(f'  Params MS:  {results.n_params}')

melhor = 'MS(2)-VAR' if results.aic < aic_ols else 'VAR linear'
print(f'\n  Modelo preferido por AIC: {melhor}')

# Plotar probabilidades suavizadas do MS-VAR
fig = plot_regime_probabilities(
    dates=df['date'].values,
    series=df['y1'].values,
    probabilities=results.smoothed_probs,
    regime_labels=['Regime 0', 'Regime 1'],
    title='MS(2)-VAR(1): Probabilidades de Regime',
)
plt.show()

# Plotar matriz de transicao
P = results.transition_matrix
print("\nMatriz de transicao:")
print(f"  {P}")
print(f"  Linhas somam 1: {[f'{P[i].sum():.4f}' for i in range(P.shape[0])]}")

fig = plot_transition_matrix(
    P,
    regime_labels=['Regime 0', 'Regime 1'],
    title='Matriz de Transicao - MS(2)-VAR(1)',
)
plt.show()

## 5. Previsao condicional ao regime

O MS-VAR permite fazer previsoes **condicionais** ao regime ativo:

$$E[\mathbf{Y}_{T+h} \mid S_{T+1} = s, \mathcal{Y}_T] = \boldsymbol{\mu}_s + \boldsymbol{\Phi}_{1,s} \mathbf{Y}_T$$

Isso e util para **analise de cenarios**:
- "Se a economia permanecer em expansao, qual e a previsao?"
- "Se entrar em recessao, como as variaveis se comportam?"

A previsao **incondicional** (ponderada por regime) e:

$$E[\mathbf{Y}_{T+h} \mid \mathcal{Y}_T] = \sum_{s=1}^{K} P(S_{T+h} = s \mid \mathcal{Y}_T) \cdot E[\mathbf{Y}_{T+h} \mid S_{T+h} = s, \mathcal{Y}_T]$$

In [ ]:
# Previsao condicional a cada regime
h = 8
Y_last = Y[-1].copy()

# Extrair mu e Phi para cada regime e calcular previsoes iterativas
forecasts = {}
for s in range(results.k_regimes):
    mu_s, phi_s, sigma_s = model._unpack_var_params(results.params, s)
    Phi_1 = phi_s[:, :n_vars]

    fc = np.zeros((h, n_vars))
    y_prev = Y_last.copy()
    for step in range(h):
        y_next = mu_s + Phi_1 @ y_prev
        fc[step] = y_next
        y_prev = y_next
    forecasts[s] = fc

# Plotar previsoes condicionais
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, var in enumerate(var_names):
    ax = axes[i]
    # Historico recente
    n_hist = 20
    ax.plot(range(-n_hist, 0), Y[-n_hist:, i], 'k-', lw=1, label='Historico')
    ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)

    # Previsoes por regime
    for s in range(results.k_regimes):
        color = regime_colors[s]
        label = f'Regime {s}'
        ax.plot(range(h), forecasts[s][:, i], color=color, marker='o',
                markersize=4, lw=2, label=label)

    ax.set_title(f'Previsao condicional: {var}', fontsize=12)
    ax.set_xlabel('Horizonte (0 = ultimo observado)')
    ax.set_ylabel(var)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Previsao Condicional ao Regime - MS(2)-VAR(1)', fontsize=14)
plt.tight_layout()
plt.show()

# Previsao incondicional (ponderada por regime)
last_probs = results.smoothed_probs[-1]
print(f"Probabilidades do ultimo periodo: {last_probs}")
print("\nPrevisao incondicional (ponderada):")
for step in range(h):
    fc_unc = sum(last_probs[s] * forecasts[s][step] for s in range(results.k_regimes))
    print(f"  h={step+1}: y1={fc_unc[0]:.4f}, y2={fc_unc[1]:.4f}")

## Conclusao

Neste notebook, aprendemos:

- A **motivacao** para modelos MS-VAR: capturar mudancas estruturais em sistemas multivariados
- Como estimar um **MS(2)-VAR(1)** com o algoritmo EM
- Como calcular **IRFs por regime**, revelando assimetrias nas respostas a choques
- Como comparar com um **VAR linear** usando AIC/BIC
- Como fazer **previsoes condicionais** a cada regime (analise de cenarios)

No proximo notebook, combinamos regime-switching com **volatilidade condicional** no modelo **MS-GARCH**.

### Referencias

- Krolzig, H.-M. (1997). *Markov-Switching Vector Autoregressions*. Springer.
- Hamilton, J.D. (1989). A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle. *Econometrica*, 57(2), 357-384.
- Ehrmann, M., Ellison, M. & Valla, N. (2003). Regime-dependent impulse response functions in a Markov-switching VAR model. *Economics Letters*, 78(3), 295-299.
- Haas, M., Mittnik, S., & Paolella, M.S. (2004). A New Approach to Markov-Switching GARCH Models. *Journal of Financial Econometrics*, 2(4), 493-530.